## South Africa Tender Procurement Analysis
* Data Cleaning

**Import Libraries & Load Raw Data**

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

print("📥 Loading raw datasets...\n")

# Load all datasets (same as before)
main = pd.read_csv("../00_raw_data/main.csv", low_memory=False)
awards = pd.read_csv("../00_raw_data/awards.csv", low_memory=False)
awards_suppliers = pd.read_csv("../00_raw_data/awards_suppliers.csv", low_memory=False)
contracts = pd.read_csv("../00_raw_data/contracts.csv", low_memory=False)
contracts_documents = pd.read_csv("../00_raw_data/contracts_documents.csv", low_memory=False)
parties = pd.read_csv("../00_raw_data/parties.csv", low_memory=False)
tender_documents = pd.read_csv("../00_raw_data/tender_documents.csv", low_memory=False)
tender_tenderers = pd.read_csv("../00_raw_data/tender_tenderers.csv", low_memory=False)

print("✅ All datasets loaded.")


📥 Loading raw datasets...

✅ All datasets loaded.


**Function to Save Cleaned Data**

In [2]:
def save_cleaned(df, filename):
    """Save cleaned dataframe to 01_cleaned_data folder"""
    path = os.path.join("../01_cleaned_data", filename)
    df.to_csv(path, index=False)
    print(f"✅ Saved: {path}")

**Quick clean Function for specific datasets based on name**

In [64]:
def quick_clean(df, name):
    """
    Quick clean for specific datasets based on name.
    Handles missing values, date conversions, dtype fixes, and standardizes column names.

    Parameters:
    df : DataFrame to clean
    name : str - dataset identifier: 'main_clean', 'awards_clean', 'parties_clean',
        'contracts_clean', 'contract_suppliers_clean', 'tender_documents_clean', 
        'tender_tenderers_clean'

    Returns:
    Cleaned DataFrame
    """
    df = df.copy()

    # 1. Standardize column names: lowercase
    df.columns = df.columns.str.lower()
    print(f"[{name}] Standardized {len(df.columns)} column names")

    # 2. Dataset-specific cleaning + dtype fixes
    if name == 'main_clean':
        total_rows = len(df)
        missing = df.isnull().sum()
        missing_pct = (missing / total_rows) * 100
        missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})

        # Drop columns with >60% missing
        cols_to_drop = missing_df[missing_df['percent'] > 60].index.tolist()
        df = df.drop(columns=cols_to_drop)
        print(f"[{name}] Dropped {len(cols_to_drop)} columns with >60% missing")

        # Handle remaining missing
        cols_to_handle = missing_df[
            (missing_df['percent'] > 0) &
            (missing_df['percent'] <= 60)
        ].index.tolist()

        for col in cols_to_handle:
            if col not in df.columns:
                continue
            pct = (df[col].isna().sum() / len(df)) * 100
            count = df[col].isna().sum()

            # <1% missing: drop rows
            if pct < 1:
                before = len(df)
                df = df.dropna(subset=[col])
                print(f"[{name}] Dropped {before - len(df)} rows for {col} | {count} = {pct:.3f}% missing")
            # 5-60% missing: impute
            elif 5 <= pct < 60:
                if df[col].dtype == 'object':
                        df[col] = df[col].fillna('Unknown')
                        print(f"[{name}] Filled {col} with 'Unknown' | {count} = {pct:.2f}% missing")
                else:
                    median_val = df[col].median()
                    df[col] = df[col].fillna(median_val)
                    print(f"[{name}] Filled {col} with median | {count} = {pct:.2f}% missing")
            # 50-60% missing: flag
            elif 50 <= pct <= 60:
                print(f"[{name}] WARNING: {col} has {pct:.2f}% missing. Manual review needed")

        # Datatype conversions for main
        date_cols = ['date', 'tender_tenderPeriod_endDate', 'tender_tenderPeriod_startDate', 
                    'tender_briefingSession_date']
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
                print(f"[{name}] Converted {col} to datetime")
                

    elif name == 'awards_clean':
        # Drop rows for description: 9/8190 = 0.11% missing
        if 'description' in df.columns:
            before = len(df)
            df = df.dropna(subset=['description'])
            print(f"[{name}] Dropped {before - len(df)} rows for description | 0.11% missing")


    elif name == 'parties_clean':
        # Drop contactpoint_faxnumber: 100% missing
        if 'contactpoint_faxnumber' in df.columns:
            df = df.drop(columns=['contactpoint_faxnumber'])
            print("[{name}] Dropped contactpoint_faxnumber: 100% missing")

        # Drop rows for contactpoint_name: 17/8190 = 0.21% missing
        if 'contactpoint_name' in df.columns:
            before = len(df)
            df = df.dropna(subset=['contactpoint_name'])
            print(f"[{name}] Dropped {before - len(df)} rows for contactpoint_name | 0.21% missing")

        # Drop rows + clean contactpoint_telephone: 46/8190 = 0.56% missing
        if 'contactpoint_telephone' in df.columns:
            before = len(df)
            df = df.dropna(subset=['contactpoint_telephone'])
            print(f"[{name}] Dropped {before - len(df)} rows for contactpoint_telephone | 0.56% missing")

    elif name == 'contracts_clean':
        if 'title' in df.columns:
            before = len(df)
            df = df.dropna(subset=['title'])
            print(f"[{name}] Dropped {before - len(df)} rows with missing title | 0.08% missing")

        # Datatype conversions
        date_cols = ['period_enddate', 'period_startdate', 'period_maxextentdate', 'datesigned']
        for col in date_cols:
            if col in df.columns:
                # Convert to datetime first
                df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')
                
                # Treat 0001-01-01 as missing - its a sentinal value
                df.loc[df[col].dt.year == 1, col] = pd.NaT
                invalid_count = df[col].isna().sum()
                print(f"[{name}] Converted {col} to datetime. Missing/Invalid: {invalid_count} rows")
                
        # Drop all rows with period dates
        period_cols = ['period_enddate', 'period_startdate', 'period_maxextentdate']
        existing_period_cols = [col for col in period_cols if col in df.columns]
        if existing_period_cols:
            before = len(df)
            df = df.dropna(subset=existing_period_cols)
            dropped = before - len(df)
            print(f"[{name}] Dropped {dropped} rows with missing period dates")
                


    elif name == 'contract_documents_clean':
        # Datatype conversions
        date_cols = ['datemodified', 'datepublished']
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')
                # Year 0001 = sentinel value, treat as NaT
                df.loc[df[col].dt.year == 1, col] = pd.NaT
                invalid_count = df[col].isna().sum()
                print(f"[{name}] Converted {col} to datetime. Missing/Invalid: {invalid_count} rows")
                
        # Drop datemodified: 96% missing makes it useless
        if 'datemodified' in df.columns:
            missing_pcts = df['datemodified'].isna().sum() / len(df) * 100
            df = df.drop(columns=['datemodified'])
            print(f"[{name}] Dropped datemodified: {missing_pcts:.1f}% missing")
            

        

    elif name == 'tender_documents_clean':
        # Datatype conversions
        date_cols = ['datemodified', 'datepublished']
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
                print(f"[{name}] Converted {col} to datetime")

    elif name == 'tender_tenderers_clean':
        # Drop 4 missing rows in name: 4/2175 = 0.18%
        if 'name' in df.columns:
            before = len(df)
            df = df.dropna(subset=['name'])
            print(f"[{name}] Dropped {before - len(df)} rows with missing name | 0.18% missing")

    print(f"[{name}] Final shape: {df.shape}, Missing values: {df.isnull().sum().sum()}\n")
    save_cleaned(df, f"{name}_cleaned.csv")
    return df


**Part A: Clean MAIN.CSV**

In [5]:
print("Starting data cleaning...\n")
main_clean = quick_clean(main, 'main_clean')

Starting data cleaning...

[main_clean] Standardized 36 column names
[main_clean] Dropped 4 columns with >60% missing
[main_clean] Dropped 1 rows for buyer_id | 1 = 0.002% missing
[main_clean] Dropped 0 rows for buyer_name | 0 = 0.000% missing
[main_clean] Dropped 24 rows for tender_title | 24 = 0.055% missing
[main_clean] Dropped 87 rows for tender_procurementmethoddetails | 87 = 0.201% missing
[main_clean] Dropped 6 rows for tender_status | 6 = 0.014% missing
[main_clean] Dropped 1 rows for tender_description | 1 = 0.002% missing
[main_clean] Filled tender_mainprocurementcategory with 'Unknown' | 9031 = 20.87% missing
[main_clean] Dropped 42 rows for tender_contactperson_name | 42 = 0.097% missing
[main_clean] Dropped 14 rows for tender_contactperson_email | 14 = 0.032% missing
[main_clean] Dropped 62 rows for tender_contactperson_telephonenumber | 62 = 0.143% missing
[main_clean] Dropped 0 rows for tender_procuringentity_id | 0 = 0.000% missing
[main_clean] Dropped 0 rows for tender

**Part B: Clean AWARDS.CSV**

In [6]:
awards_clean = quick_clean(awards, 'awards_clean')

[awards_clean] Standardized 8 column names
[awards_clean] Dropped 9 rows for description | 0.11% missing
[awards_clean] Final shape: (8181, 8), Missing values: 0

✅ Saved: ../01_cleaned_data\awards_clean_cleaned.csv


**Part C: Clean AWARDS_SUPPLIERS.CSV**

In [7]:
awards_suppliers_clean = quick_clean(awards_suppliers, 'awards_suppliers_clean')

[awards_suppliers_clean] Standardized 5 column names
[awards_suppliers_clean] Final shape: (8190, 5), Missing values: 0

✅ Saved: ../01_cleaned_data\awards_suppliers_clean_cleaned.csv


**Part D: Clean CONTRACTS**

In [8]:
contracts_clean = quick_clean(contracts, 'contracts_clean')

[contracts_clean] Standardized 14 column names
[contracts_clean] Dropped 1 rows with missing title | 0.08% missing
[contracts_clean] Converted period_enddate to datetime. Missing/Invalid: 172 rows
[contracts_clean] Converted period_startdate to datetime. Missing/Invalid: 171 rows
[contracts_clean] Converted period_maxextentdate to datetime. Missing/Invalid: 172 rows
[contracts_clean] Converted datesigned to datetime. Missing/Invalid: 0 rows
[contracts_clean] Dropped 172 rows with missing period dates
[contracts_clean] Final shape: (1057, 14), Missing values: 0

✅ Saved: ../01_cleaned_data\contracts_clean_cleaned.csv


**Part E: Clean CONTRACT_DOCUMENTS.CSV**

In [9]:
contract_documents_clean = quick_clean(contracts_documents, 'contract_documents_clean')

[contract_documents_clean] Standardized 11 column names
[contract_documents_clean] Converted datemodified to datetime. Missing/Invalid: 1182 rows
[contract_documents_clean] Converted datepublished to datetime. Missing/Invalid: 0 rows
[contract_documents_clean] Dropped datemodified: 96.1% missing
[contract_documents_clean] Final shape: (1230, 10), Missing values: 0

✅ Saved: ../01_cleaned_data\contract_documents_clean_cleaned.csv


**PART F: Cean PARTIES.CSV**

In [10]:
parties_clean = quick_clean(parties, 'parties_clean')

[parties_clean] Standardized 12 column names
[{name}] Dropped contactpoint_faxnumber: 100% missing
[parties_clean] Dropped 17 rows for contactpoint_name | 0.21% missing
[parties_clean] Dropped 31 rows for contactpoint_telephone | 0.56% missing
[parties_clean] Final shape: (8142, 11), Missing values: 0

✅ Saved: ../01_cleaned_data\parties_clean_cleaned.csv


**Part G: Clean TENDER_DOCUMENTS.CSV**

In [11]:
tender_documents_clean = quick_clean(tender_documents, 'tender_documents_clean')

[tender_documents_clean] Standardized 12 column names
[tender_documents_clean] Converted datemodified to datetime
[tender_documents_clean] Converted datepublished to datetime
[tender_documents_clean] Final shape: (44313, 12), Missing values: 0

✅ Saved: ../01_cleaned_data\tender_documents_clean_cleaned.csv


**Part H: Clean TENDER_TENDERERS.CSV**

In [12]:
tender_tenderers_clean = quick_clean(tender_tenderers, 'tender_tenderers_clean')

[tender_tenderers_clean] Standardized 5 column names
[tender_tenderers_clean] Dropped 4 rows with missing name | 0.18% missing
[tender_tenderers_clean] Final shape: (2175, 5), Missing values: 0

✅ Saved: ../01_cleaned_data\tender_tenderers_clean_cleaned.csv
